<h1 style = 'color: red'>Assignment: SQL Notebook</h1>

In [1]:
%load_ext sql

In [5]:
import csv, sqlite3
import prettytable
prettytable.DEFAULT = 'DEFAULT'
con = sqlite3.connect('my_data1.db')
cur = con.cursor()
%sql sqlite:///my_data1.db
import pandas as pd

In [6]:
df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv')
# method = 'multi': means bulk insertion
df.to_sql('SPACEXTBL', con, if_exists = 'replace', index = False, method = 'multi')

101

In [7]:
%%sql
DROP TABLE IF EXISTS SPACEXTABLE;
CREATE TABLE SPACEXTABLE AS 
SELECT * FROM SPACEXTBL 
WHERE Date IS NOT NULL;

 * sqlite:///my_data1.db
Done.
Done.


[]

<h3 style = 'color: yellow'>Task1: display the names of the unique launch sites in the space mission</h3>

In [8]:
%%sql
SELECT DISTINCT Launch_Site
FROM SPACEXTABLE;

 * sqlite:///my_data1.db
Done.


Launch_Site
CCAFS LC-40
VAFB SLC-4E
KSC LC-39A
CCAFS SLC-40


<h3 style = 'color: yellow'>Task2: display 5 records where launch sites begin with the string 'CCA'</h3>

In [17]:
%%sql
/* 
use of LIKE to have a certain string-like restriction
LIMIT is to limit the records that displays
*/
SELECT * FROM SPACEXTABLE WHERE Launch_Site LIKE 'CCA%' LIMIT 5;

 * sqlite:///my_data1.db
Done.


Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of Brouere cheese",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


<h3 style = 'color: yellow'>Task3: display the total payload mass carried by boosters launched by NASA (CRS)</h3>

In [19]:
%%sql
SELECT SUM(PAYLOAD_MASS__KG_) AS total_payload_mass
FROM SPACEXTABLE
WHERE Customer = 'NASA (CRS)';

 * sqlite:///my_data1.db
Done.


total_payload_mass
45596


<h3 style = 'color: yellow'>Task4: display average payload mass carried by booster version F9 v1.1</h3>

In [20]:
%%sql
SELECT AVG(PAYLOAD_MASS__KG_) AS average_payload_mass
FROM SPACEXTABLE
WHERE Booster_Version = 'F9 v1.1'

 * sqlite:///my_data1.db
Done.


average_payload_mass
2928.4


<h3 style = 'color: yellow'>Task5: list the date when the first successful ladning outcome in ground pad was achieved</h3>

In [22]:
%%sql
SELECT Date
FROM SPACEXTABLE
WHERE Landing_Outcome = 'Success (ground pad)'
LIMIT 1;

 * sqlite:///my_data1.db
Done.


Date
2015-12-22


<h3 style = 'color: yellow'>Task6: List the name of the boosters which have success in drope ship and have payload mass greater than 4000 but less than 6000</h3>

In [24]:
%%sql
SELECT DISTINCT(Booster_Version)
FROM SPACEXTABLE
WHERE Landing_Outcome = 'Success (drone ship)' 
AND PAYLOAD_MASS__KG_ BETWEEN 4000 AND 6000;

 * sqlite:///my_data1.db
Done.


Booster_Version
F9 FT B1022
F9 FT B1026
F9 FT B1021.2
F9 FT B1031.2


<h3 style = 'color: yellow'>Task7: list the total number of successful and failure mission outcomes</h3>

In [27]:
%%sql
SELECT COUNT(*)
FROM SPACEXTABLE
WHERE Landing_Outcome LIKE 'Failure%';

 * sqlite:///my_data1.db
Done.


COUNT(*)
10


In [28]:
%%sql
SELECT COUNT(*)
FROM SPACEXTABLE
WHERE Landing_Outcome LIKE 'Success%';

 * sqlite:///my_data1.db
Done.


COUNT(*)
61


In [29]:
%%sql
SELECT
SUM(CASE WHEN Landing_Outcome LIKE 'Failure%' THEN 1 ELSE 0 END) AS Failure_Count,
SUM(CASE WHEN Landing_Outcome LIKE 'Success%' THEN 1 ELSE 0 END) AS Success_Count
FROM SPACEXTABLE;

 * sqlite:///my_data1.db
Done.


Failure_Count,Success_Count
10,61


<h3 style = 'color: yellow'>Task8: list all the booster_versions that have carried the maximum payload mass, using a subquery with a suitable aggregate function</h3>

In [31]:
%%sql
SELECT *
FROM SPACEXTABLE
WHERE (Booster_Version, PAYLOAD_MASS__KG_) IN (
    SELECT Booster_Version, MAX(PAYLOAD_MASS__KG_)
    FROM SPACEXTABLE
    GROUP BY Booster_Version
    )
LIMIT 1;

 * sqlite:///my_data1.db
Done.


Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)


In [32]:
%%sql
SELECT Booster_Version, MAX(PAYLOAD_MASS__KG_) AS Max_Payload_Mass
FROM SPACEXTABLE
GROUP BY Booster_Version;

 * sqlite:///my_data1.db
Done.


Booster_Version,Max_Payload_Mass
F9 B4 B1039.2,2647
F9 B4 B1040.2,5384
F9 B4 B1041.2,9600
F9 B4 B1043.2,6460
F9 B4 B1039.1,3310
F9 B4 B1040.1,4990
F9 B4 B1041.1,9600
F9 B4 B1042.1,3500
F9 B4 B1043.1,5000
F9 B4 B1044,6092


In [33]:
%%sql
SELECT S.*
FROM SPACEXTABLE S
JOIN (
    SELECT Booster_Version, MAX(PAYLOAD_MASS__KG_) AS MaxPayLoad
    FROM SPACEXTABLE
    GROUP BY Booster_Version
    ) AS M
ON S.Booster_Version = M.Booster_Version AND S.PAYLOAD_MASS__KG_ = M.MaxPayLoad;

 * sqlite:///my_data1.db
Done.


Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of Brouere cheese",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt
2013-09-29,16:00:00,F9 v1.1 B1003,VAFB SLC-4E,CASSIOPE,500,Polar LEO,MDA,Success,Uncontrolled (ocean)
2014-08-05,8:00:00,F9 v1.1,CCAFS LC-40,AsiaSat 8,4535,GTO,AsiaSat,Success,No attempt
2014-09-07,5:00:00,F9 v1.1 B1011,CCAFS LC-40,AsiaSat 6,4428,GTO,AsiaSat,Success,No attempt
2014-09-21,5:52:00,F9 v1.1 B1010,CCAFS LC-40,SpaceX CRS-4,2216,LEO (ISS),NASA (CRS),Success,Uncontrolled (ocean)
2015-01-10,9:47:00,F9 v1.1 B1012,CCAFS LC-40,SpaceX CRS-5,2395,LEO (ISS),NASA (CRS),Success,Failure (drone ship)


<h3 style = 'color: yellow'>Task9: list the records which will display the month names, failure landing outcomes in drone ship, booster versions, launch site for the months in year 2015</h3>

In [35]:
%%sql
SELECT 
    SUBSTR(Date, 6, 2) AS Month,
    SUBSTR(Date, 0, 5) AS Year,
    Landing_Outcome,
    Booster_Version,
    Launch_Site
FROM SPACEXTABLE
WHERE Landing_Outcome LIKE '%drone ship%' AND Year = '2015';

 * sqlite:///my_data1.db
Done.


Month,Year,Landing_Outcome,Booster_Version,Launch_Site
01,2015,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
04,2015,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40
06,2015,Precluded (drone ship),F9 v1.1 B1018,CCAFS LC-40


<h3 style = 'color: yellow'>Task10: rank the count of landing outcomes (such as failure (drone ship) or success (ground pad) between the date 2010-06-04 and 2017-03-20, in descending order</h3>

In [50]:
%%sql
SELECT COUNT(*) AS COUNT, Landing_Outcome
FROM SPACEXTABLE
WHERE Landing_Outcome IN ('Failure (drone ship)', 'Success (ground pad)')
AND Date BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY Landing_Outcome
ORDER BY COUNT DESC;

 * sqlite:///my_data1.db
Done.


COUNT,Landing_Outcome
5,Failure (drone ship)
3,Success (ground pad)
